In [ ]:
DROP = False
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"
start_date = "2020-06-01"

In [ ]:
# df = df_sp500
df = df.loc["2020-05-31":"2022-06-01"]
df.index = pd.to_datetime(df.index)
df.drop(columns=["ticker"],inplace=True)
if start_date:
    df = df[df.index >= start_date]

In [ ]:
df = df["VIX"]

In [ ]:
predictions_rf = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Predictions/predictions_rf.pkl")
directions_rf = predictions_rf -  predictions_rf.shift()

In [ ]:
directions_rf["2022-01-19"]

In [ ]:
df = pd.merge(directions_rf,df, left_index=True, right_index=True)

In [ ]:
df

In [ ]:
# df = df["2022-01-13":]

In [ ]:
import requests
import pandas as pd
import numpy as np
from math import floor
from termcolor import colored as cl
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (20, 10)
plt.style.use('fivethirtyeight')

# def get_historical_data(symbol, start_date = None):
#     # api_key = open(r'api_key.txt')
#     # api_url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&apikey={api_key}&outputsize=full'
#     # raw_df = requests.get(api_url).json()
#     # df = pd.DataFrame(df[f'Time Series (Daily)']).T
#     # df = df.rename(columns = {'1. open': 'open', '2. high': 'high', '3. low': 'low', '4. VIX': 'VIX', '5. adjusted VIX': 'adj VIX', '6. volume': 'volume'})
#     # for i in df.columns:
#     #     df[i] = df[i].astype(float)
#     # df = df.iloc[::-1].drop(['7. dividend amount', '8. split coefficient'], axis = 1)
#     if start_date:
#         df = df[df.index >= start_date]
#     return df

# googl = get_historical_data('GOOGL', '2020-01-01')
# googl

def get_vix(price, slow, fast, smooth):
    exp1 = price.ewm(span = fast, adjust = False).mean()
    exp2 = price.ewm(span = slow, adjust = False).mean()
    vix = pd.DataFrame(exp1 - exp2).rename(columns = {'VIX':'vix'})
    signal = pd.DataFrame(vix.ewm(span = smooth, adjust = False).mean()).rename(columns = {'vix':'signal'})
    hist = pd.DataFrame(vix['vix'] - signal['signal']).rename(columns = {0:'hist'})
    frames =  [vix, signal, hist]
    df = pd.concat(frames, join = 'inner', axis = 1)
    return df

df_vix = get_vix(df['VIX'], 26, 12, 9)
df_vix

def plot_vix(prices, vix, signal, hist):
    ax1 = plt.subplot2grid((8,1), (0,0), rowspan = 5, colspan = 1)
    ax2 = plt.subplot2grid((8,1), (5,0), rowspan = 3, colspan = 1)

    ax1.plot(prices)
    ax2.plot(vix, color = 'grey', linewidth = 1.5, label = 'MACD')
    ax2.plot(signal, color = 'skyblue', linewidth = 1.5, label = 'SIGNAL')

    for i in range(len(prices)):
        if str(hist[i])[0] == '-':
            ax2.bar(prices.index[i], hist[i], color = '#ef5350')
        else:
            ax2.bar(prices.index[i], hist[i], color = '#26a69a')

    plt.legend(loc = 'lower right')

plot_vix(df['VIX'], df_vix['vix'], df_vix['signal'], df_vix['hist'])

def implement_vix_strategy(prices, data):    
    buy_price = []
    sell_price = []
    vix_signal = []
    signal = 0

    for i in range(len(data)):
        if data['predicted_value'][i] > 0:
            if signal != 1:
                buy_price.append(prices[i])
                sell_price.append(np.nan)
                signal = 1
                vix_signal.append(signal)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
        elif data['predicted_value'][i] < 0:
            if signal != -1:
                buy_price.append(np.nan)
                sell_price.append(prices[i])
                signal = -1
                vix_signal.append(signal)
            else:
                buy_price.append(np.nan)
                sell_price.append(np.nan)
                vix_signal.append(0)
        else:
            buy_price.append(np.nan)
            sell_price.append(np.nan)
            vix_signal.append(0)
            
    return buy_price, sell_price, vix_signal
            
buy_price, sell_price, vix_signal = implement_vix_strategy(df['VIX'], df)

ax1 = plt.subplot2grid((8,1), (0,0), rowspan = 5, colspan = 1)
ax2 = plt.subplot2grid((8,1), (5,0), rowspan = 3, colspan = 1)

ax1.plot(df['VIX'], color = 'skyblue', linewidth = 2, label = 'Random Forest VIX')
ax1.plot(df.index, buy_price, marker = '^', color = 'green', markersize = 10, label = 'BUY SIGNAL', linewidth = 0)
ax1.plot(df.index, sell_price, marker = 'v', color = 'r', markersize = 10, label = 'SELL SIGNAL', linewidth = 0)
ax1.legend()
ax1.set_title('Random Forest VIX Strategy')
ax2.plot(df_vix['vix'], color = 'grey', linewidth = 1.5, label = 'MACD')
ax2.plot(df_vix['signal'], color = 'skyblue', linewidth = 1.5, label = 'SIGNAL')

for i in range(len(df_vix)):
    if str(df_vix['hist'][i])[0] == '-':
        ax2.bar(df_vix.index[i], df_vix['hist'][i], color = '#ef5350')
    else:
        ax2.bar(df_vix.index[i], df_vix['hist'][i], color = '#26a69a')
        
plt.legend(loc = 'lower right')
plt.show()

position = []
for i in range(len(vix_signal)):
    if vix_signal[i] > 1:
        position.append(0)
    else:
        position.append(1)
        
for i in range(len(df['VIX'])):
    if vix_signal[i] == 1:
        position[i] = 1
    elif vix_signal[i] == -1:
        position[i] = 0
    else:
        position[i] = position[i-1]
        
vix = df_vix['vix']
signal = df_vix['signal']
VIX_price = df['VIX']
vix_signal = pd.DataFrame(vix_signal).rename(columns = {0:'vix_signal'}).set_index(df.index)
position = pd.DataFrame(position).rename(columns = {0:'vix_position'}).set_index(df.index)

frames = [VIX_price, vix, signal, vix_signal, position]
strategy = pd.concat(frames, join = 'inner', axis = 1)

strategy

df_ret = pd.DataFrame(np.diff(df['VIX'])).rename(columns = {0:'returns'})
vix_strategy_ret = []

for i in range(len(df_ret)):
    try:
        returns = df_ret['returns'][i]*strategy['vix_position'][i]
        vix_strategy_ret.append(returns)
    except:
        pass
    
vix_strategy_ret_df = pd.DataFrame(vix_strategy_ret).rename(columns = {0:'vix_returns'})

investment_value = 100000
number_of_stocks = floor(investment_value/df['VIX'][0])
vix_investment_ret = []

for i in range(len(vix_strategy_ret_df['vix_returns'])):
    returns = number_of_stocks*vix_strategy_ret_df['vix_returns'][i]
    vix_investment_ret.append(returns)

vix_investment_ret_df = pd.DataFrame(vix_investment_ret).rename(columns = {0:'investment_returns'})
total_investment_ret = round(sum(vix_investment_ret_df['investment_returns']), 2)
profit_percentage = floor((total_investment_ret/investment_value)*100)
print(cl('Profit gained from the Random Forest strategy by investing $100k in df : {}'.format(total_investment_ret), attrs = ['bold']))
print(cl('Profit percentage of the Random Forest strategy : {}%'.format(profit_percentage), attrs = ['bold']))

def get_benchmark(start_date, investment_value):
    spy = df['VIX']#get_historical_data('SPY', start_date)['VIX']
    benchmark = pd.DataFrame(np.diff(spy)).rename(columns = {0:'benchmark_returns'})
    
    investment_value = investment_value
    number_of_stocks = floor(investment_value/spy[0])
    benchmark_investment_ret = []
    
    for i in range(len(benchmark['benchmark_returns'])):
        returns = number_of_stocks*benchmark['benchmark_returns'][i]
        benchmark_investment_ret.append(returns)

    benchmark_investment_ret_df = pd.DataFrame(benchmark_investment_ret).rename(columns = {0:'investment_returns'})
    return benchmark_investment_ret_df

benchmark = get_benchmark('2020-06-01', 100000)

investment_value = 100000
total_benchmark_investment_ret = round(sum(benchmark['investment_returns']), 2)
benchmark_profit_percentage = floor((total_benchmark_investment_ret/investment_value)*100)
print(cl('Benchmark profit by investing $100k : {}'.format(total_benchmark_investment_ret), attrs = ['bold']))
print(cl('Benchmark Profit percentage : {}%'.format(benchmark_profit_percentage), attrs = ['bold']))
print(cl('Random Forest Strategy profit is {}% higher than the Benchmark Profit'.format(profit_percentage - benchmark_profit_percentage), attrs = ['bold']))